<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Classification trade-offs

**[Classification trade-offs](https://learning.nextia-ai.com/courses/math/m06/classification-trade-offs/)** · Essential Mathematics and Statistics for AI · Module 6, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** count the confusion matrix of ten tickets and of the 1,000 test tickets, calculate precision, recall, the false-positive rate and accuracy, see why "flag nothing" is 92% accurate, and move the threshold to see the trade-off.

| | |
|---|---|
| **Time** | About 25 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | How to read a count as a share ("3 of 4 is 75%"), from [Conditional probability: Probability inside a group](https://learning.nextia-ai.com/courses/math/m03/conditional-probability/#probability-inside-a-group). |
| **Tested** | Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m06/classification-trade-offs/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M06-L01-classification-trade-offs/classification-trade-offs-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

## Setup: the scores of the 1,000 test tickets

The next cell defines two lists: `urgent_scores`, the model's scores of the 80 urgent test tickets, and `normal_scores`, the scores of the 920 tickets that are not urgent. They are the same seeded scores that the lesson's explorer draws. The data is fictional.

Run the cell. You should see `Ready: 80 urgent and 920 not urgent tickets, 1000 in total`.

In [ ]:
# The scores, written in full (you do not need to read them).
urgent_scores = [
    0.828, 0.694, 0.812, 0.634, 0.732, 0.326, 0.654, 0.66, 0.382, 0.674, 0.747, 0.951, 0.867, 0.507, 0.59, 0.678,
    0.469, 0.788, 0.777, 0.379, 0.701, 0.708, 0.505, 0.867, 0.58, 0.578, 0.527, 0.573, 0.684, 0.77, 0.426, 0.566,
    0.55, 0.746, 0.665, 0.538, 0.842, 0.536, 0.593, 0.655, 0.904, 0.551, 0.605, 0.766, 0.727, 0.486, 0.47, 0.693,
    0.526, 0.615, 0.895, 0.844, 0.296, 0.775, 0.308, 0.88, 0.753, 0.576, 0.565, 0.645, 0.372, 0.589, 0.446, 0.555,
    0.517, 0.579, 0.877, 0.707, 0.665, 0.356, 0.993, 0.426, 0.42, 0.633, 0.587, 0.462, 0.824, 0.813, 0.681, 0.575,
]
normal_scores = [
    0.153, 0.358, 0.566, 0.22, 0.364, 0.461, 0.137, 0.223, 0.403, 0.099, 0.213, 0.57, 0.437, 0.369, 0.431, 0.182,
    0.128, 0.525, 0.043, 0.201, 0.281, 0.36, 0.352, 0.282, 0.289, 0.033, 0.344, 0.119, 0.348, 0.586, 0.206, 0.137,
    0.048, 0.303, 0.178, 0.383, 0.478, 0.24, 0.09, 0.302, 0.332, 0.284, 0.291, 0.38, 0.19, 0.103, 0.388, 0.401,
    0.071, 0.371, 0.083, 0.23, 0.104, 0.381, 0.377, 0.048, 0.285, 0.317, 0.425, 0.38, 0.216, 0.13, 0.126, 0.501,
    0.196, 0.424, 0.195, 0.241, 0.653, 0.368, 0.418, 0.062, 0.465, 0.394, 0.455, 0.398, 0.239, 0.056, 0.085, 0.281,
    0.31, 0.255, 0.087, 0.073, 0.337, 0.116, 0.022, 0.415, 0.345, 0.412, 0.357, 0.393, 0.703, 0.524, 0.223, 0.258,
    0.303, 0.03, 0.307, 0.441, 0.153, 0.35, 0.34, 0.303, 0.127, 0.179, 0.257, 0.529, 0.142, 0.294, 0.283, 0.304,
    0.298, 0.236, 0.477, 0.108, 0.052, 0.574, 0.102, 0.456, 0.317, 0.274, 0.219, 0.464, 0.178, 0.381, 0.16, 0.168,
    0.402, 0.241, 0.302, 0.127, 0.326, 0.185, 0.291, 0.392, 0.224, 0.259, 0.443, 0.176, 0.248, 0.712, 0.269, 0.418,
    0.33, 0.084, 0.197, 0.265, 0.21, 0.02, 0.418, 0.42, 0.163, 0.488, 0.211, 0.173, 0.257, 0.258, 0.368, 0.281,
    0.048, 0.427, 0.239, 0.483, 0.158, 0.112, 0.283, 0.355, 0.414, 0.133, 0.244, 0.288, 0.281, 0.113, 0.333, 0.285,
    0.132, 0.274, 0.069, 0.134, 0.087, 0.274, 0.386, 0.468, 0.332, 0.358, 0.436, 0.255, 0.315, 0.311, 0.347, 0.119,
    0.133, 0.186, 0.435, 0.023, 0.092, 0.016, 0.643, 0.364, 0.476, 0.4, 0.056, 0.428, 0.203, 0.391, 0.255, 0.355,
    0.111, 0.102, 0.208, 0.446, 0.054, 0.287, 0.331, 0.404, 0.07, 0.47, 0.233, 0.007, 0.085, 0.32, 0.115, 0.125,
    0.443, 0.412, 0.078, 0.366, 0.186, 0.094, 0.173, 0.268, 0.139, 0.52, 0.167, 0.251, 0.306, 0.105, 0.231, 0.267,
    0.059, 0.236, 0.234, 0.315, 0.142, 0.146, 0.128, 0.228, 0.219, 0.203, 0.388, 0.147, 0.125, 0.441, 0.526, 0.063,
    0.053, 0.401, 0.34, 0.313, 0.32, 0.095, 0.242, 0.18, 0.151, 0.208, 0.312, 0.212, 0.494, 0.341, 0.346, 0.274,
    0.272, 0.321, 0.197, 0.259, 0.009, 0.165, 0.135, 0.207, 0.623, 0.26, 0.134, 0.333, 0.152, 0.316, 0.211, 0.344,
    0.129, 0.334, 0.105, 0.335, 0.176, 0.183, 0.015, 0.206, 0.162, 0.254, 0.386, 0.343, 0.595, 0.218, 0.311, 0.251,
    0.118, 0.139, 0.112, 0.253, 0.282, 0.382, 0.425, 0.426, 0.355, 0.207, 0.403, 0.307, 0.282, 0.252, 0.193, 0.213,
    0.274, 0.189, 0.398, 0.136, 0.153, 0.075, 0.416, 0.61, 0.324, 0.172, 0.212, 0.541, 0.128, 0.214, 0.368, 0.112,
    0.173, 0.108, 0.013, 0.328, 0.16, 0.194, 0.2, 0.231, 0.205, 0.142, 0.26, 0.209, 0.227, 0.118, 0.464, 0.206,
    0.259, 0.649, 0.392, 0.231, 0.202, 0.167, 0.318, 0.255, 0.383, 0.476, 0.519, 0.327, 0.289, 0.153, 0.295, 0.112,
    0.178, 0.302, 0.244, 0.345, 0.487, 0.062, 0.344, 0.296, 0.241, 0.378, 0.213, 0.433, 0.32, 0.462, 0.279, 0.177,
    0.098, 0.434, 0.175, 0.292, 0.077, 0.348, 0.387, 0.348, 0.294, 0.191, 0.169, 0.24, 0.397, 0.092, 0.379, 0.288,
    0.283, 0.308, 0.304, 0.096, 0.269, 0.075, 0.579, 0.195, 0.18, 0.269, 0.213, 0.274, 0.075, 0.069, 0.099, 0.365,
    0.209, 0.19, 0.296, 0.126, 0.066, 0.294, 0.187, 0.338, 0.095, 0.109, 0.481, 0.056, 0.236, 0.081, 0.341, 0.159,
    0.295, 0.261, 0.409, 0.33, 0.304, 0.106, 0.066, 0.267, 0.307, 0.431, 0.19, 0.299, 0.505, 0.38, 0.375, 0.366,
    0.316, 0.12, 0.471, 0.374, 0.421, 0.183, 0.346, 0.258, 0.127, 0.439, 0.308, 0.226, 0.222, 0.237, 0.214, 0.11,
    0.027, 0.197, 0.115, 0.374, 0.298, 0.433, 0.035, 0.061, 0.377, 0.289, 0.26, 0.215, 0.285, 0.006, 0.483, 0.148,
    0.097, 0.121, 0.048, 0.336, 0.152, 0.261, 0.376, 0.315, 0.186, 0.35, 0.217, 0.323, 0.193, 0.043, 0.352, 0.306,
    0.38, 0.25, 0.446, 0.579, 0.097, 0.225, 0.235, 0.462, 0.082, 0.463, 0.287, 0.149, 0.267, 0.291, 0.242, 0.297,
    0.379, 0.165, 0.375, 0.178, 0.37, 0.207, 0.162, 0.32, 0.126, 0.466, 0.139, 0.193, 0.023, 0.437, 0.422, 0.539,
    0.153, 0.297, 0.093, 0.413, 0.255, 0.282, 0.161, 0.377, 0.243, 0.237, 0.009, 0.231, 0.454, 0.274, 0.079, 0.086,
    0.089, 0.116, 0.412, 0.342, 0.324, 0.139, 0.143, 0.23, 0.173, 0.224, 0.186, 0.468, 0.304, 0.104, 0.27, 0.046,
    0.324, 0.317, 0.302, 0.306, 0.338, 0.722, 0.277, 0.248, 0.345, 0.115, 0.315, 0.368, 0.288, 0.128, 0.143, 0.337,
    0.153, 0.312, 0.144, 0.078, 0.363, 0.203, 0.226, 0.207, 0.051, 0.215, 0.184, 0.284, 0.311, 0.285, 0.1, 0.136,
    0.081, 0.221, 0.272, 0.457, 0.117, 0.373, 0.277, 0.248, 0.223, 0.264, 0.067, 0.407, 0.283, 0.34, 0.203, 0.395,
    0.179, 0.134, 0.208, 0.219, 0.352, 0.263, 0.23, 0.178, 0.31, 0.217, 0.369, 0.405, 0.311, 0.308, 0.509, 0.286,
    0.335, 0.061, 0.466, 0.282, 0.242, 0.393, 0.425, 0.08, 0.091, 0.1, 0.11, 0.35, 0.38, 0.168, 0.477, 0.204,
    0.13, 0.234, 0.298, 0.327, 0.219, 0.552, 0.329, 0.404, 0.206, 0.248, 0.353, 0.312, 0.087, 0.269, 0.242, 0.102,
    0.254, 0.337, 0.218, 0.111, 0.212, 0.297, 0.176, 0.17, 0.263, 0.413, 0.542, 0.328, 0.248, 0.304, 0.388, 0.375,
    0.332, 0.029, 0.431, 0.266, 0.025, 0.376, 0.171, 0.373, 0.216, 0.471, 0.343, 0.363, 0.522, 0.285, 0.166, 0.196,
    0.346, 0.182, 0.456, 0.362, 0.336, 0.244, 0.297, 0.151, 0.278, 0.249, 0.411, 0.158, 0.096, 0.065, 0.108, 0.074,
    0.274, 0.512, 0.373, 0.495, 0.513, 0.293, 0.311, 0.102, 0.48, 0.499, 0.339, 0.24, 0.228, 0.47, 0.271, 0.067,
    0.117, 0.3, 0.526, 0.124, 0.413, 0.229, 0.108, 0.248, 0.306, 0.168, 0.209, 0.202, 0.343, 0.36, 0.453, 0.47,
    0.239, 0.19, 0.071, 0.355, 0.125, 0.468, 0.254, 0.219, 0.304, 0.263, 0.262, 0.047, 0.52, 0.18, 0.462, 0.397,
    0.312, 0.244, 0.097, 0.333, 0.24, 0.243, 0.489, 0.28, 0.314, 0.186, 0.051, 0.356, 0.293, 0.143, 0.167, 0.172,
    0.069, 0.137, 0.303, 0.514, 0.329, 0.143, 0.109, 0.201, 0.385, 0.426, 0.204, 0.412, 0.031, 0.363, 0.062, 0.056,
    0.211, 0.348, 0.325, 0.346, 0.168, 0.37, 0.109, 0.257, 0.163, 0.455, 0.364, 0.18, 0.457, 0.162, 0.303, 0.044,
    0.22, 0.275, 0.122, 0.159, 0.352, 0.464, 0.229, 0.121, 0.4, 0.094, 0.358, 0.218, 0.509, 0.37, 0.248, 0.082,
    0.401, 0.26, 0.302, 0.236, 0.127, 0.537, 0.166, 0.009, 0.49, 0.322, 0.217, 0.144, 0.229, 0.12, 0.341, 0.029,
    0.015, 0.107, 0.707, 0.199, 0.508, 0.303, 0.372, 0.225, 0.211, 0.156, 0.217, 0.097, 0.238, 0.125, 0.385, 0.342,
    0.427, 0.146, 0.284, 0.257, 0.289, 0.109, 0.51, 0.254, 0.206, 0.038, 0.146, 0.467, 0.273, 0.519, 0.182, 0.167,
    0.247, 0.554, 0.302, 0.203, 0.411, 0.527, 0.015, 0.244, 0.289, 0.068, 0.04, 0.404, 0.244, 0.131, 0.127, 0.474,
    0.347, 0.171, 0.176, 0.322, 0.23, 0.476, 0.204, 0.221, 0.327, 0.305, 0.076, 0.728, 0.011, 0.535, 0.262, 0.135,
    0.129, 0.136, 0.212, 0.223, 0.071, 0.23, 0.606, 0.009, 0.454, 0.246, 0.198, 0.136, 0.425, 0.101, 0.312, 0.31,
    0.404, 0.24, 0.216, 0.434, 0.325, 0.115, 0.638, 0.578,
]
print(f"Ready: {len(urgent_scores)} urgent and {len(normal_scores)} not urgent tickets, {len(urgent_scores) + len(normal_scores)} in total")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. From a score to a flag: ten tickets

The model gives each ticket a **score** from 0 to 1. If the score is at least the **threshold**, the ticket is **flagged** and a senior agent reviews it. Here are the ten tickets from the lesson, with the threshold 0.50. Grace checked each ticket by hand, so `urgent` is the correct answer.

> **Predict.** Of the 4 urgent tickets, how many are flagged? Of the 6 tickets that are not urgent, how many are flagged? Then run the cell.

In [ ]:
ten = [  # (ticket, score, really urgent?)
    ("T-5001", 0.91, True), ("T-5002", 0.74, False), ("T-5003", 0.66, True), ("T-5004", 0.58, False),
    ("T-5005", 0.52, True), ("T-5006", 0.45, True), ("T-5007", 0.33, False), ("T-5008", 0.21, False),
    ("T-5009", 0.12, False), ("T-5010", 0.05, False),
]
threshold = 0.50

def outcome(flagged, urgent):
    if urgent:
        return "caught" if flagged else "missed"
    return "false alarm" if flagged else "correctly left"

rows = []
for ticket, score, urgent in ten:
    flagged = score >= threshold
    rows.append([ticket, score, "yes" if flagged else "no", "yes" if urgent else "no", outcome(flagged, urgent)])
table(["Ticket", "Score", "Flagged?", "Urgent?", "What happened"], rows)

> **Check.** You should see 3 tickets caught, 1 missed (T-5006, score 0.45), 2 false alarms and 4 correctly left.

Every ticket lands in one of four groups. The table of the four counts is the **confusion matrix**. Run the cell to count the groups. You should see 3 and 1 in the urgent row, and 2 and 4 in the not urgent row.

In [ ]:
outcomes = [outcome(score >= threshold, urgent) for _, score, urgent in ten]
caught, missed = outcomes.count("caught"), outcomes.count("missed")
false_alarms, left = outcomes.count("false alarm"), outcomes.count("correctly left")
table(["", "Flagged", "Not flagged", "Total"], [
    ["Really urgent", caught, missed, caught + missed],
    ["Not urgent", false_alarms, left, false_alarms + left],
    ["Total", caught + false_alarms, missed + left, len(ten)],
])

## 2. Three questions, three totals

Grace, Omar and Tomás each ask a different question. Each answer divides by a different total. Always ask: "out of what?"

- **Grace:** of the flagged tickets, how many are urgent? This is **precision**.
- **Omar:** of the urgent tickets, how many are flagged? This is **recall**.
- **Tomás:** of the tickets that are not urgent, how many are flagged by mistake? This is the **false-positive rate**.

> **Predict.** Which of the three is the largest for the ten tickets? Then run the cell. You should see 60.0%, 75.0% and 33.3%.

In [ ]:
print(f"precision:           {caught} of {caught + false_alarms} = {caught / (caught + false_alarms):.1%}")
print(f"recall:              {caught} of {caught + missed} = {caught / (caught + missed):.1%}")
print(f"false-positive rate: {false_alarms} of {false_alarms + left} = {false_alarms / (false_alarms + left):.1%}")

## 3. Worked example: the 1,000 test tickets

Ten tickets are too few to judge a model. Priya uses the 1,000 test tickets from the setup: 80 are urgent (8%) and 920 are not urgent. Professionals call the four groups **true positives** (TP, caught), **false negatives** (FN, missed), **false positives** (FP, false alarms) and **true negatives** (TN, correctly left).

The next cell defines `count(threshold)`, which returns the four counts, and `thresholds`, the 101 thresholds of the explorer. Then it counts at 0.50.

> **Predict.** The lesson's table at 0.50 has 112 flagged tickets. Then run the cell. You should see TP 65, FN 15, FP 47 and TN 873, and the totals 80, 920, 112 and 1000.

In [ ]:
def count(threshold):
    """The four counts at a threshold. A ticket is flagged when its score is at least the threshold."""
    tp = len([s for s in urgent_scores if s >= threshold])   # urgent, flagged
    fp = len([s for s in normal_scores if s >= threshold])   # not urgent, flagged
    fn = len(urgent_scores) - tp                              # urgent, missed
    tn = len(normal_scores) - fp                              # not urgent, correctly left
    return tp, fn, fp, tn

# The 101 thresholds of the explorer: 0.00, 0.01, ..., 1.00
thresholds = [round(i / 100, 2) for i in range(101)]

tp, fn, fp, tn = count(0.50)
table(["", "Flagged", "Not flagged", "Total"], [
    ["Urgent", tp, fn, tp + fn],
    ["Not urgent", fp, tn, fp + tn],
    ["Total", tp + fp, fn + tn, tp + fn + fp + tn],
])

**Accuracy** is the share of all tickets where the model is right: (TP + TN) out of all tickets. Run the cell. You should see a precision of 58.0%, a recall of 81.25% (81.3% on the page), a false-positive rate of 5.1% and an accuracy of 93.8%.

In [ ]:
def shares(tp, fn, fp, tn):
    """Precision, recall, false-positive rate and accuracy, in percent. Precision is None when nothing is flagged."""
    precision = tp / (tp + fp) * 100 if tp + fp else None
    recall = tp / (tp + fn) * 100
    fpr = fp / (fp + tn) * 100
    accuracy = (tp + tn) / (tp + fn + fp + tn) * 100
    return precision, recall, fpr, accuracy

precision, recall, fpr, accuracy = shares(tp, fn, fp, tn)
table(["Measure", "Out of what?", "Result (%)"], [
    ["Precision", f"the {tp + fp} flagged tickets", precision],
    ["Recall", f"the {tp + fn} urgent tickets", recall],
    ["False-positive rate", f"the {fp + tn} tickets that are not urgent", fpr],
    ["Accuracy", "all 1000 tickets", accuracy],
])

## 4. How accuracy misleads with a rare class

Priya compares three ways to flag the tickets: flag nothing, threshold 0.50 and threshold 0.70. "Flag nothing" is the same as a threshold above every score, for example 1.00.

> **Predict.** Which of the three has the highest accuracy? Which misses the most urgent tickets? Then run the cell.

In [ ]:
rules = [("Flag nothing", 1.00), ("Threshold 0.50", 0.50), ("Threshold 0.70", 0.70)]
labels, acc, missed = [], [], []
for name, t in rules:
    tp, fn, fp, tn = count(t)
    labels.append(name)
    acc.append(round(shares(tp, fn, fp, tn)[3], 1))
    missed.append(fn)
print("Accuracy (%)")
bars(labels, acc)
print()
print("Urgent tickets missed (of 80)")
bars(labels, missed)

> **Check.** You should see an accuracy of 92.0, 93.8 and 94.2: three bars of almost the same length. The missed urgent tickets are 80, 15 and 53: very different bars. The rule with the highest accuracy misses more than three times as many urgent tickets as threshold 0.50.

When one class is rare, accuracy mostly measures the common class. The share of the rare class is the **base rate**, here 8%. Always state the base rate next to accuracy.

## 5. Move the threshold

On the lesson page, the explorer draws the scores of the two classes and moves the threshold. Here you see the same scores as text. The next cell counts the scores of each class in bins of 0.1. Each class has its own bar scale, because the not urgent class is more than 11 times bigger.

> **Predict.** Between which scores do the two classes overlap? Then run the cell.

In [ ]:
labels = [f"{i / 10:.1f}–{(i + 1) / 10:.1f}" for i in range(10)]
def per_bin(scores):
    return [len([s for s in scores if i / 10 <= s < (i + 1) / 10]) for i in range(10)]
print("Not urgent (920 tickets)")
bars(labels, per_bin(normal_scores), width=30)
print()
print("Urgent (80 tickets)")
bars(labels, per_bin(urgent_scores), width=30)

> **Check.** You should see the not urgent tickets mostly from 0.0 to 0.5 (the largest bar is 253 at 0.2–0.3), and the urgent tickets mostly from 0.5 to 0.8 (the largest bar is 22 at 0.5–0.6). Both classes have tickets from 0.2 to 0.8, and most of the overlap is from about 0.3 to 0.7.

Now count at eleven thresholds. Run the cell. At 1.00 nothing is flagged, so precision is undefined: the cell prints `-`.

> **Predict.** When the threshold goes up, does precision go up or down? And recall?

In [ ]:
rows = []
for t in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    tp, fn, fp, tn = count(t)
    precision, recall, fpr, accuracy = shares(tp, fn, fp, tn)
    rows.append([f"{t:.2f}", tp, fn, fp, tn, "-" if precision is None else round(precision, 1), round(recall, 1), round(accuracy, 1)])
table(["Threshold", "TP", "FN", "FP", "TN", "Precision %", "Recall %", "Accuracy %"], rows, digits=1)

> **Check.** You should see recall fall from 100.0% at 0.00 to 0.0% at 1.00, and precision rise from 8.0% at 0.00 to 100.0% at 0.80. (Python rounds 81.25 to 81.2; the lesson page shows 81.3%.) You cannot move the threshold to make both better: that is the **trade-off**. Only a better model, with less overlap, improves both.

> **Your turn: change one thing.** In the cell above, replace the list of thresholds with `[0.40, 0.50]` and run it again. Compare your result with knowledge check 3 on the lesson page.

## 6. Guided practice: a lower threshold

Grace asks: "What if we flag more, at threshold 0.30?" The counts are TP = 79, FN = 1, FP = 363 and TN = 557.

> **Your turn.** Replace each `...` with a calculation from the four counts. Give the shares in percent, for example `50.0`. Then run the check cell.

In [ ]:
tp, fn, fp, tn = 79, 1, 363, 557

flagged = ...
precision = ...            # in percent
recall = ...               # in percent
false_positive_rate = ...  # in percent
accuracy = ...             # in percent

Run the check cell.

In [ ]:
expect("the number of flagged tickets", flagged, 442)
expect("the precision", precision, 17.9, tol=0.05)
expect("the recall", recall, 98.8, tol=0.05)
expect("the false-positive rate", false_positive_rate, 39.5, tol=0.05)
expect("the accuracy", accuracy, 63.6, tol=0.05)

Accuracy fell from 93.8% to 63.6%, but the model now misses only 1 urgent ticket. Your senior agents would see 442 flags per 1,000 tickets, and fewer than 1 in 5 is urgent. Write your sentence for Grace in `c03-notes.md`. Whether this is better depends on the costs: that is the next lesson.

## 7. Independent variation: a stricter threshold

Tomás wants fewer flags, at threshold **0.70**.

> **Your turn.** This time, get the four counts from the scores with `count(0.70)`. Then calculate precision, recall, the false-positive rate and accuracy, in percent. Then run the check cell. Write one sentence for Omar in `c03-notes.md` that does not use the word "accuracy".

In [ ]:
counts_70 = ...           # the four counts (tp, fn, fp, tn) at 0.70, from count()

precision_70 = ...
recall_70 = ...
false_positive_rate_70 = ...
accuracy_70 = ...

Run the check cell.

In [ ]:
expect("the counts at 0.70", counts_70, (27, 53, 5, 915))
expect("the precision at 0.70", precision_70, 84.4, tol=0.05)
expect("the recall at 0.70", recall_70, 33.8, tol=0.05)
expect("the false-positive rate at 0.70", false_positive_rate_70, 0.5, tol=0.05)
expect("the accuracy at 0.70", accuracy_70, 94.2, tol=0.05)

Almost every flag is real, but the model misses about two of every three urgent tickets. The example answer for Omar is on the lesson page.

## 8. Failure case: accuracy without class balance

A report says "94% accurate", and the team releases the model at threshold 0.70. The next cell writes the report that Priya asks for: the base rate first, then the confusion matrix and recall, next to the "flag nothing" rule.

> **Predict.** How many more urgent tickets does the model catch than "flag nothing"? Then run the cell.

In [ ]:
def report(t):
    tp, fn, fp, tn = count(t)
    precision, recall, fpr, accuracy = shares(tp, fn, fp, tn)
    print(f"Threshold {t:.2f}: base rate {tp + fn} of {tp + fn + fp + tn} tickets are urgent")
    print(f"  TP {tp}, FN {fn}, FP {fp}, TN {tn}")
    print(f"  recall {recall:.1f}%, precision {'-' if precision is None else f'{precision:.1f}%'}, accuracy {accuracy:.1f}%")

report(0.70)
report(1.00)   # flag nothing

> **Check.** You should see that the model at 0.70 catches 27 urgent tickets and "flag nothing" catches 0, but their accuracy differs by only 2.2 points (94.2% and 92.0%). The report with the base rate and recall shows that the model misses 53 of 80 urgent tickets. **Fix:** report the base rate, the confusion matrix, and recall and precision for the rare class, and compare with "flag nothing".

## Check your understanding and recap

Answer the **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/math/m06/classification-trade-offs/). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you counted the four groups of a confusion matrix, for ten tickets and for the 1,000 test tickets. Precision divides by the flags, recall by the urgent tickets, and the false-positive rate by the tickets that are not urgent. "Flag nothing" is 92% accurate, so accuracy alone hides the missed urgent tickets. A higher threshold raises precision and lowers recall.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Thresholds and costs](https://learning.nextia-ai.com/courses/math/m06/thresholds-and-costs/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m06/classification-trade-offs/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.